In [1]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from tqdm import tqdm
import ast
import numpy as np
from pathlib import Path

In [2]:
# read market news data
data_path = Path("../Dataset")
df = pd.read_csv(data_path / 'market_news_data.csv')
print(df.head(5))
print(df.tail(5))

              timestamp                                            symbols  \
0  2025-12-31T22:46:16Z                                           ['COST']   
1  2025-12-31T22:30:46Z                                           ['NVDA']   
2  2025-12-31T22:06:21Z  ['ARKG', 'ARKK', 'BEAM', 'CRSP', 'NTLA', 'PACB...   
3  2025-12-31T21:38:43Z               ['BTCUSD', 'DOGEUSD', 'NKE', 'TSLA']   
4  2025-12-31T21:13:34Z                                           ['AVGO']   

                                            headline  \
0  Here's How Much You Would Have Made Owning Cos...   
1  If You Invested $100 In NVIDIA Stock 5 Years A...   
2  Cathie Wood's Biotech Trend: Twist, Beam, CRIS...   
3  From Spike In SPY, QQQ Call Volume To Musk's S...   
4  Options Corner: Why Broadcom's Pullback Statis...   

                                             summary  
0                                                     
1                                                     
2  The massive rotation curre

/var/folders/lv/z5s79n1j3fs87jp5qd8_r4dm0000gn/T/ipykernel_4306/1540209762.py:3: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path / 'market_news_data.csv')


In [3]:
# 1. Setup Device
device = "mps"

# 2. Load FinBERT
model_name = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Create the sentiment pipeline
nlp = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, device=device)

# 3. Prepar text for FinBERT
def combine_text(row):
    headline = str(row['headline'])
    summary = str(row['summary'])
    
    # If summary is 'nan', 'NaN', or empty, just use the headline
    if not summary or summary.lower() == 'nan' or len(summary.strip()) == 0:
        return headline
    
    # Otherwise, join them (FinBERT handles up to 512 tokens)
    return f"{headline}. {summary}"

processed_text = df.apply(combine_text, axis=1).tolist()

# 4. Batch processing
batch_size = 64
results = []

for i in tqdm(range(0, len(processed_text), batch_size)):
    batch = processed_text[i : i + batch_size]
    # Truncation is critical here since adding summaries makes the text longer
    batch_results = nlp(batch, truncation=True, max_length=512)
    results.extend(batch_results)

# 5. Extract results back to DataFrame
df['sentiment_label'] = [res['label'] for res in results]
df['sentiment_score'] = [res['score'] for res in results]

print(df.head(5))
print(df.tail(5))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
100%|██████████| 2623/2623 [24:26<00:00,  1.79it/s]


              timestamp                                            symbols  \
0  2025-12-31T22:46:16Z                                           ['COST']   
1  2025-12-31T22:30:46Z                                           ['NVDA']   
2  2025-12-31T22:06:21Z  ['ARKG', 'ARKK', 'BEAM', 'CRSP', 'NTLA', 'PACB...   
3  2025-12-31T21:38:43Z               ['BTCUSD', 'DOGEUSD', 'NKE', 'TSLA']   
4  2025-12-31T21:13:34Z                                           ['AVGO']   

                                            headline  \
0  Here's How Much You Would Have Made Owning Cos...   
1  If You Invested $100 In NVIDIA Stock 5 Years A...   
2  Cathie Wood's Biotech Trend: Twist, Beam, CRIS...   
3  From Spike In SPY, QQQ Call Volume To Musk's S...   
4  Options Corner: Why Broadcom's Pullback Statis...   

                                             summary sentiment_label  \
0                                                            neutral   
1                                                 

In [9]:
df = df[['timestamp', 'symbols', 'sentiment_label', 'sentiment_score']]
df.head(5)

,timestamp,symbols,sentiment_label,sentiment_score
0,2025-12-31T22:46:16Z,['COST'],neutral,0.935342
1,2025-12-31T22:30:46Z,['NVDA'],neutral,0.899845
2,2025-12-31T22:06:21Z,"['ARKG', 'ARKK', 'BEAM', 'CRSP', 'NTLA', 'PACB...",neutral,0.882337
3,2025-12-31T21:38:43Z,"['BTCUSD', 'DOGEUSD', 'NKE', 'TSLA']",neutral,0.912637
4,2025-12-31T21:13:34Z,['AVGO'],negative,0.929533


In [ ]:
df.to_csv(data_path / "market_news_with_sentiment.csv", index=False)